In [17]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3"
)

In [73]:
def send_email_to_crm(x):
    print("Sending email to CRM system...")
    x["status"] = "Email Sent"
    return x


def add_to_db(x):
    print("Adding user to the database...")
    x["status"] = "User Added to DB"
    return x
    


In [74]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableBranch

send_email = RunnableLambda(lambda x: send_email_to_crm(x))
add_user = RunnableLambda(lambda x: add_to_db(x))

conditional_chain = RunnableBranch(
    (lambda x: x.action == "send_email", send_email),
    (lambda x: x.action == "add_user", add_user),
    RunnablePassthrough()
)



In [75]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
    "You are a feedback classifier. Analyze the sentiment of the user feedback "
        "and output exactly one word from the following options: 'send_email', 'add_user', or 'none'.\n\n"
        "Logic:\n"
        "- Negative (dissatisfied/complaint) -> send_email\n"
        "- Positive (satisfied/compliment) -> add_user\n"
        "- Neutral/Unclear -> none"
    """,
    input_variables=["feedback"]
)


In [76]:
from pydantic import BaseModel, Field

class outputState(BaseModel):
    action: str = Field(description="The action to be taken based on user feedback", default="none")
    

In [77]:
structured_model = model.with_structured_output(outputState)


In [88]:
chain = prompt | structured_model | conditional_chain



In [93]:
chain.invoke({"feedback": "I don't like the product. this is bad."})

Sending email to CRM system...


TypeError: 'outputState' object does not support item assignment

In [85]:
res

outputState(action='none')